 # Sci/Tech News Q&A with Gemma (RAG)

**Track:** Build w Gemma — Gemma Challenge (Google Developer Group) · Hack Fiesta Miami, Aug 2026

This notebook implements a retrieval-augmented generation (RAG) pipeline:

1. Load the **AG News** dataset from Hugging Face and keep only the **Sci/Tech** label.
2. Chunk + embed the articles with a small sentence-embedding model.
3. Build a similarity index (FAISS) over the embeddings.
4. Retrieve the most relevant chunks for a user question.
5. Feed the retrieved context to **Gemma** (`google/gemma-2-2b-it`, via Hugging Face `transformers`) and generate a **cited** answer.

In [7]:
# 1. Install dependencies
!pip install -q -U transformers accelerate datasets sentence-transformers faiss-cpu bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 76.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.7 MB/s eta 0:00:00:00:01


In [8]:
# 2. Authenticate with Hugging Face using Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# This line fetches the *actual* token value stored inside your "gemma-rag-token" secret
hf_token = UserSecretsClient().get_secret("gemma-rag-token")

# Now log in using that fetched token
login(token=hf_token)
print("Successfully logged in to Hugging Face Hub!")

Successfully logged in to Hugging Face Hub!


## 3. Load AG News and keep only Sci/Tech

AG News labels: `0=World, 1=Sports, 2=Business, 3=Sci/Tech`.
We load it straight from the Hugging Face `datasets` hub (no manual download).


In [9]:
from datasets import load_dataset

SCI_TECH_LABEL = 3
N_ARTICLES = 300  # keep the notebook fast; raise this if you want a bigger corpus

raw = load_dataset("ag_news", split="test")  # 7,600 rows total
scitech = raw.filter(lambda r: r["label"] == SCI_TECH_LABEL)
scitech = scitech.select(range(min(N_ARTICLES, len(scitech))))

print(f"Loaded {len(scitech)} Sci/Tech articles out of {len(raw)} total AG News test rows.")
scitech[0]

Filter:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loaded 300 Sci/Tech articles out of 7600 total AG News test rows.


{'text': 'The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\\privately funded suborbital space flight, has officially announced the first\\launch date for its manned rocket.',
 'label': 3}

In [10]:
# AG News packs "Title \\ short description" into one `text` field — split them
# apart so we have clean, citable titles.

def split_title_body(example):
    parts = example["text"].split(" \\ ", 1)
    title = parts[0].strip()
    body = parts[1].strip() if len(parts) > 1 else example["text"]
    return {"title": title, "body": body}

scitech = scitech.map(split_title_body)
articles = [
    {"id": f"art-{i}", "title": r["title"], "text": r["text"]}
    for i, r in enumerate(scitech)
]
articles[:3]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

[{'id': 'art-0',
  'title': 'The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\\privately funded suborbital space flight, has officially announced the first\\launch date for its manned rocket.',
  'text': 'The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\\privately funded suborbital space flight, has officially announced the first\\launch date for its manned rocket.'},
 {'id': 'art-1',
  'title': 'Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop a method of producing better peptides, which are short chains of amino acids, the building blocks of proteins.',
  'text': 'Ky

## 4. Chunk + embed the corpus

Articles here are already short, so "chunking" mostly means treating each article as one
chunk. We embed with `all-MiniLM-L6-v2` (fast, CPU-friendly) — **not** Gemma itself, since
Gemma is a text-generation model and doesn't expose an embeddings API. This mirrors the
production app's split between an embedding model and a generation model.

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [a["text"] for a in articles]
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")
print("Embedding matrix shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding matrix shape: (300, 384)


In [12]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity via inner product on normalized vectors
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dim}.")

FAISS index built with 300 vectors of dimension 384.


In [13]:
def retrieve(question: str, top_k: int = 5):
    q_emb = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        a = articles[idx]
        results.append({"title": a["title"], "text": a["text"], "score": float(score)})
    return results

# quick sanity check
for r in retrieve("AI chip funding announcement", top_k=3):
    print(f"{r['score']:.3f}  {r['title']}")

0.472  Intel Delays Launch of Projection TV Chip In another product postponement, semiconductor giant Intel Corp. said it won't be offering a chip for projection TVs by the end of 2004 as it had announced earlier this year.
0.435  Intel drops prices on computer chips SAN FRANCISCO - Intel Corp. has cut prices on its computer chips by as much as 35 percent, though analysts on Monday said the cuts were probably unrelated to swelling inventories of the world #39;s largest chip maker.
0.406  Intel Shrinks Transistor Size By 30 pinkUZI writes  quot;Intel will announce that it has crammed 500 million transistors on to a single memory chip, shrinking them in size by 30.


## 5. Load Gemma for generation

`google/gemma-2-2b-it` is small enough to run on a single Kaggle T4 GPU in 4-bit
(via `bitsandbytes`). Swap in a larger variant (e.g. `google/gemma-2-9b-it`) if your
runtime has more memory.

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEMMA_MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Loaded {GEMMA_MODEL_ID} on {model.device}.")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded google/gemma-2-2b-it on cpu.


In [15]:
def gemma_generate(system_prompt, user_prompt, max_new_tokens=256):
    # Format the prompt using Gemma's chat template or standard instruction format
    chat = [
        {"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}
    ]
    
    # Apply the tokenizer chat template
    formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    
    # Tokenize and explicitly send to the model's device
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # Generate the response (Note the **inputs unpacking here!)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3
        )
    
    # Decode only the generated text (skipping the prompt part)
    input_len = inputs["input_ids"].shape[1]
    generated_tokens = output_ids[0][input_len:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return answer

## 6. The RAG loop: retrieve → augment → generate (Gemma)

The system prompt forces Gemma to answer **only** from the numbered context and to
say so explicitly when the answer isn't in the retrieved articles — this is the
citation-or-refusal pattern used to keep answers grounded and reduce hallucination.

In [16]:
SYSTEM_PROMPT = (
    "You are a fact-checking assistant for Sci/Tech news. "
    "Answer ONLY using the numbered context snippets provided. "
    "Cite snippet numbers like [1], [2] that support each claim. "
    "If the context does not contain the answer, say so plainly instead of guessing."
)

def rag_answer(question: str, top_k: int = 5) -> dict:
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(f"[{i+1}] ({h['title']}) {h['text']}" for i, h in enumerate(hits))
    user_prompt = f"Context snippets from AG News (Sci/Tech):\n\n{context}\n\nQuestion: {question}\n\nAnswer:"
    answer = gemma_generate(SYSTEM_PROMPT, user_prompt)
    return {"question": question, "answer": answer, "sources": hits}

In [17]:
# 7. Try it out
result = rag_answer("What recent Sci/Tech stories mention AI chips or semiconductors?")

print("Q:", result["question"])
print("\nGemma's answer:\n", result["answer"])
print("\nSources considered:")
for i, s in enumerate(result["sources"], 1):
    print(f"  [{i}] ({s['score']:.3f}) {s['title']}")

Q: What recent Sci/Tech stories mention AI chips or semiconductors?

Gemma's answer:
 [1], [2], [3], [4], [5] 


Sources considered:
  [1] (0.476) AMD Ships Notebook Chips It wasn #39;t the first to go small, and it won #39;t be the biggest producer, but AMD #39;s (Quote, Chart) 64-bit 90-nanometer (nm) chips are expected to make waves in the semiconductor pool.
  [2] (0.454) Toyota reports a silicon carbide breakthrough Move over silicon chips, there is a new semiconductor king on the horizon. Silicon carbide #39;s (SiC) potential has been known since the 1950 #39;s, but the properties that make is attractive also make it hard to work with.
  [3] (0.424) IBM Chips May Someday Heal Themselves New technology applies electrical fuses to help identify and repair faults.
  [4] (0.417) Intel drops prices on computer chips SAN FRANCISCO - Intel Corp. has cut prices on its computer chips by as much as 35 percent, though analysts on Monday said the cuts were probably unrelated to swelling inve

In [18]:
# A few more example queries to demonstrate grounded, cited answers
for q in [
    "Have there been any router or network security vulnerabilities reported?",
    "What's happening with space launches or satellites?",
    "Is there any news about cryptocurrency regulation?",  # likely NOT in a small Sci/Tech sample -> should refuse
]:
    r = rag_answer(q, top_k=4)
    print("="*80)
    print("Q:", r["question"])
    print(r["answer"])

Q: Have there been any router or network security vulnerabilities reported?
[1] Cisco Systems issued a security advisory warning that some networks using its routers may be vulnerable to denial-of-service attacks. 
[2]  Yevgeny Kaspersky has raised concerns of a major attack on the internet today. 
[4] Industry cyber security standards fail to reach some of the most vulnerable components of the power grid. 


Therefore, the answer is yes. 

Q: What's happening with space launches or satellites?
The provided text discusses several different aspects of space launches and satellites, but doesn't provide a comprehensive overview. 

Here's a breakdown of what we can glean:

* **Shuttle Safety:** NASA is working to improve the safety of the space shuttle before it resumes visits to the International Space Station. [1]
* **Da Vinci Project:** The Da Vinci Project, a group in Toronto, is facing challenges with paperwork and insurance for their homemade, manned spacecraft launch in October. [3]

## 8. Export the curated corpus for the companion Node.js app

The Node/Vercel app in this submission normally pulls AG News live from the Hugging Face
`datasets-server` REST API at request time. Exporting this notebook's already-filtered,
already-cleaned `articles` list lets the app **ingest this exact curated corpus** instead —
useful for reproducibility (the app serves precisely what was validated here) and for
offline/judging scenarios with unreliable Wi-Fi.


In [19]:
from datasets import load_dataset

# 1. Load the AG News dataset from Hugging Face
print("Loading AG News dataset...")
dataset = load_dataset("ag_news", split="train")

# 2. Filter for Sci/Tech label (AG News label 3 is Sci/Tech)
# We'll take a subset (e.g., 1000 articles) to keep FAISS running fast and smooth
scitech_subset = dataset.filter(lambda x: x["label"] == 3).select(range(1000))

# 3. Format into the 'articles' list your export (and RAG pipeline) expects
articles = []
for item in scitech_subset:
    articles.append({
        "title": item["text"].split(".")[0], # or extract title if formatted differently
        "text": item["text"]
    })

print(f"Created 'articles' list with {len(articles)} items successfully!")

Loading AG News dataset...
Created 'articles' list with 1000 items successfully!


In [20]:
import os

file_path = 'scitech_articles_export.json'
if os.path.exists(file_path):
    size = os.path.getsize(file_path)
    print(f"File size: {size} bytes")
else:
    print("File does not exist!")

File size: 90 bytes


In [21]:
import json

# Assuming your data is stored in a variable called `articles` or `scitech_data`
# (Replace `scitech_data` with whatever variable name contains your actual data)
scitech_data = [
    {"title": "Example Article", "content": "Sample content..."}
]  # <--- Replace this with your actual variable!

# Write data safely with encoding and indentation
with open('scitech_articles_export.json', 'w', encoding='utf-8') as f:
    json.dump(scitech_data, f, ensure_ascii=False, indent=4)

print('Successfully wrote data to scitech_articles_export.json!')

Successfully wrote data to scitech_articles_export.json!


In [22]:
from IPython.display import FileLink

FileLink('scitech_articles_export.json')

/kaggle/working/scitech_articles_export.json